# System One models on the pricer: Jev vs Kev vs Nimble vs Laya

Ed's `lab.ipynb` compares frontier LLMs with Jev, TypeSafe's hosted System One decision model. This notebook adds the open alternatives to the same test:

| model | who | how it runs | open weights |
|---|---|---|---|
| **Jev** | TypeSafe | OpenRouter Decisions API | no |
| **Kev-4B** | Jared Palmer | OpenRouter Decisions API (also self-hostable) | yes, Apache-2.0 |
| **Nimble** (9B) | Bespoke Labs | Ollama, locally | yes |
| **Laya** (421M) | Convai Innovations | in this notebook, locally | yes, Apache-2.0 |
| **Laya, fine-tuned** | trained here | in this notebook, locally | yes |

Plus Luna, the frontier LLM from Ed's notebook, as the reference point.

The question in Ed's notebook is whether a decision model can match an LLM. The question here is one step further: **can a small open model, fine-tuned on your own data in half an hour on a laptop, beat the hosted ones?**

Every System One model gets Ed's best method, `jev_two_pass_median`: ask for a fine and a coarse price bucket in one call, combine the two distributions, take the median. The shared code is in `system_one.py`.

## Setup

Beyond `uv sync` at the repo root:

```bash
uv pip install "laya==0.3.21" nbclient
ollama pull nimble        # needs Ollama 0.35+, ~9.5 GB
```

`.env` needs `OPENROUTER_API_KEY` (Jev, Kev, Luna) and `HF_TOKEN` (the dataset), as in Ed's README. Running everything costs about $0.04 in API calls. Fine-tuning Laya takes ~30 min on an Apple-silicon Mac (M4 Pro here) or a free Colab/Kaggle GPU.

In [1]:
import os
import sys

import pandas as pd
import plotly.io as pio
from dotenv import load_dotenv
from huggingface_hub import login
from litellm import completion
from openrouter import OpenRouter

sys.path.insert(0, os.path.abspath("../.."))  # the repo root, for Ed's pricer package
from pricer.evaluator import Tester
from pricer.items import Item

import system_one as s1

pio.renderers.default = "notebook_connected"  # keeps the saved notebook small
load_dotenv(override=True)
login(os.environ["HF_TOKEN"], add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
train, val, test = Item.from_hub("ed-donner/items_full")
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

openrouter = OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"], server_url="https://openrouter.ai")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


Ed's `evaluate()` prints the charts; `Tester` is the class behind it, and keeping each one lets us build a single comparison table at the end. Same 200 test items for every model.

In [3]:
results = {}


def run(name, predictor, workers=5):
    tester = Tester(predictor, test, title=name, workers=workers)
    tester.run()
    results[name] = tester

## The reference points: Luna and Jev

Luna exactly as in Ed's notebook, and Jev with Ed's two-pass median.

In [4]:
def luna(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    response = completion(model="openrouter/openai/gpt-5.6-luna", messages=[{"role": "user", "content": message}],
                          seed=42, reasoning_effort="none")
    return response.choices[0].message.content, response._hidden_params["response_cost"]


run("Luna", luna)

  0%|          | 0/200 [00:00<?, ?it/s]

$10 $64 $25 $30 $10 

$90 

$54 $65 $6 $120 

$37 

...

$22 $25 $2 

$5 $1 

$5 $191 $30 

$10 $9 

$60 $21 $6 
Total API cost $0.0059 for 200 items, 38.4s wall clock with 5 workers


In [5]:
run("Jev", s1.openrouter_predictor(openrouter, "typesafe/jev-1.13"))

  0%|          | 0/200 [00:00<?, ?it/s]

$19 $77 $25 $37 $1 $187 

$85 $66 $4 $86 $475 $28 $11 $21 

$30 $6 $50 $8 $82 $45 

$78 $64 $4 $120 $143 $200 

...

$1 $4 $95 $14 $55 

$135 $29 $113 $64 $66 $11 

$45 $33 $27 $1 $70 $69 $1 

$107 $43 $82 $30 $28 $19 

$1 
Total API cost $0.0194 for 200 items, 8.6s wall clock with 5 workers


## Kev-4B

Jared Palmer's open-source decision models: a LoRA adapter and a small pointer head on Qwen3.5, at 0.8B, 4B, 9B and 27B. OpenRouter hosts the 4B on the same Decisions API as Jev, so it gets exactly Jev's question.

In [6]:
run("Kev-4B", s1.openrouter_predictor(openrouter, "jaredpalmer/kev-4b"))

  0%|          | 0/200 [00:00<?, ?it/s]

$16 $87 $14 $20 $38 

$131 $75 $37 $4 $93 

$485 $74 $19 $76 $18 

$9 $28 $1 $85 $41 

...

$14 $7 $213 

$43 $32 

$285 $13 $187 

$58 $69 

$16 $5 
Total API cost $0.0134 for 200 items, 56.7s wall clock with 5 workers


## Nimble, locally through Ollama

A 9B decision model from Bespoke Labs, fine-tuned from Qwen3.5-9B. Ollama serves it on a local `/v1/systemone` endpoint with Jev's wire format. Two differences:

- Nimble allows at most 26 options per choice question, so its fine grid is **$40 wide (25 buckets)** rather than Jev's $20 (50). A slightly coarser question.
- One worker: the default 5 parallel requests would queue on the single local GPU and inflate the latency.

In [7]:
run("Nimble", s1.nimble_predictor(), workers=1)

  0%|          | 0/200 [00:00<?, ?it/s]

$92 

$33 

$16 

$10 

...

$76 

$51 

$90 

$3 

$8 
Total API cost $0.0000 for 200 items, 692.5s wall clock with 1 workers


## Laya, zero-shot

Laya is a 421M-parameter encoder, about 1/20th of Nimble's size, and it runs right here in the notebook. It fits ~192 tokens of question, so like Nimble it gets a $40 fine grid; 25 bare labels fit whole, while 50 would be cut down to 4 tokens each.

No training yet: this is what the published model knows about prices out of the box.

In [8]:
import laya

laya_base = laya.load(s1.LAYA_MODEL)
run("Laya (zero-shot)", s1.laya_predictor(laya_base), workers=1)
del laya_base

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

.venv/lib/python3.12/site-packages/laya/agent.py:1404: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


  0%|          | 0/200 [00:00<?, ?it/s]

$167 $197 $358 

...

$167 $182 $160 

$220 $315 $104 

$162 $481 $140 

$217 $189 $119 

$364 $238 
Total API cost $0.0000 for 200 items, 16.8s wall clock with 1 workers


## Fine-tuning Laya on Ed's training data

Laya ships no `train()` function; its official recipe is a hand-written PyTorch loop. `s1.train_laya` is a compact single-device version:

- Each item becomes two training rows, one per question, labelled with the bucket its price falls in.
- **RLCD**: the logits are the mean of a Gaussian policy; we sample a group of noisy probability reports and reward each with Laya's `proper_reward`, a strictly proper scoring rule. Those are maximised only by honest probabilities, so training pushes towards *calibrated* confidence. The group mean is the REINFORCE baseline. (Dividing by the group's std as well, a common GRPO trick, made training diverge: it rewards ever-sharper logits.)
- Cross-entropy alongside, as in Laya's recipe.

6,000 of the 800,000 training items, one epoch. Then `s1.calibrate_laya` fits a temperature per question on validation items. This matters here: the base checkpoint ships an invalid 0.10 for questions with 11+ options, which Laya overrides to 0.5 with a warning, and the median reads the whole distribution, so the temperature moves the estimates.

Skipped if the checkpoint already exists.

In [9]:
CHECKPOINT = "checkpoints/laya-pricer"

if not os.path.exists(os.path.join(CHECKPOINT, "model.safetensors")):
    s1.train_laya(train, val, CHECKPOINT, size=6000, epochs=1)
    s1.calibrate_laya(CHECKPOINT, val)

In [10]:
laya_tuned = laya.load(CHECKPOINT)
run("Laya (fine-tuned)", s1.laya_predictor(laya_tuned), workers=1)

  0%|          | 0/200 [00:00<?, ?it/s]

$89 $1 $26 

$18 $26 $195 

$102 $42 $5 

$175 $576 $298 

...

$68 $68 $17 

$0 $10 $72 

$22 $47 $51 

$22 $12 $110 

$7 $1 
Total API cost $0.0000 for 200 items, 16.5s wall clock with 1 workers


## The comparison

In [11]:
import numpy as np
from sklearn.metrics import r2_score


def summarise(tester):
    return {
        "average error ($)": np.mean(tester.errors),
        "r² (%)": r2_score(tester.truths, tester.guesses) * 100,
        "cost per call ($)": np.mean(tester.costs),
        "latency (s)": np.mean(tester.latencies),
    }


table = pd.DataFrame({name: summarise(tester) for name, tester in results.items()}).T
table.sort_values("average error ($)").style.format(
    {"average error ($)": "{:.2f}", "r² (%)": "{:.1f}", "cost per call ($)": "{:.6f}", "latency (s)": "{:.2f}"})

,average error ($),r² (%),cost per call ($),latency (s)
Luna,51.63,47.9,0.000030,0.95
Jev,52.77,67.8,0.000097,0.21
Nimble,68.10,38.6,0.000000,3.46
Laya (fine-tuned),70.85,28.0,0.000000,0.08
Kev-4B,73.79,45.2,0.000067,1.40
Laya (zero-shot),219.97,-191.6,0.000000,0.08


Latency is per call as the evaluator measures it: wall-clock inside the predictor. Network models include the round trip; Nimble and Laya run on the same machine, one call at a time.

## Optional: publish the fine-tuned model to Hugging Face

Training took half an hour; publishing lets anyone `laya.load()` your model without repeating it. The steps:

1. **A write token.** At huggingface.co → Settings → Access Tokens, create a token with *write* access. The read token used for the dataset can't create repos.
2. **A model card.** The repo's `README.md` is what people see first: what the model is, how it was trained, how well it does, how to load it, and the licence. Laya is Apache-2.0, so a fine-tune can be published under it too.
3. **Create the repo and upload the folder.** `huggingface_hub` does both. `laya.load("your-name/laya-pricer")` reads the same layout we saved: `rl_agent_config.json`, `model.safetensors`, `tokenizer/`, `encoder/`.

A public upload can be copied and cached the moment it's live, so check the card before you push. Set `HF_REPO` to run it; it does nothing while it's `None`.

In [12]:
HF_REPO = None  # e.g. "your-username/laya-pricer"
PRIVATE = True  # start private; flip it on the Hub once you've checked the page

if HF_REPO:
    from huggingface_hub import HfApi

    tuned = summarise(results["Laya (fine-tuned)"])
    card = f"""---
license: apache-2.0
base_model: {s1.LAYA_MODEL}
datasets: [ed-donner/items_full]
tags: [laya, system-one, decision-model, price-estimation]
---

# Laya fine-tuned for retail price estimation

[Laya]({"https://huggingface.co/" + s1.LAYA_MODEL}) (revision `{s1.LAYA_REVISION[:8]}`), fine-tuned with RLCD on
6,000 items from [ed-donner/items_full](https://huggingface.co/datasets/ed-donner/items_full) to pick the price
bucket of a product from its description.

Average error on 200 test items: **${tuned["average error ($)"]:.2f}** (two-pass median, $40 and $100 grids).

```python
import laya

agent = laya.load("{HF_REPO}")
```

See [the training notebook](https://github.com/ed-donner/pricer/tree/main/community_contributions/xianguan).
"""
    with open(os.path.join(CHECKPOINT, "README.md"), "w") as f:
        f.write(card)

    api = HfApi()
    api.create_repo(HF_REPO, repo_type="model", private=PRIVATE, exist_ok=True)
    api.upload_folder(folder_path=CHECKPOINT, repo_id=HF_REPO, repo_type="model")
    print(f"https://huggingface.co/{HF_REPO}")